> **Disclaimer:** This notebook is for *educational and research training purposes only*. It does not constitute medical, clinical, or diagnostic advice. All computational predictions (pLDDT, RMSD, affinity scores) are approximations from machine learning models and must not be treated as experimental results. See the full [DISCLAIMER](https://github.com/JobAiReady/lagos-bio-design/blob/main/DISCLAIMER.md) and [PRIVACY POLICY](https://github.com/JobAiReady/lagos-bio-design/blob/main/PRIVACY_POLICY.md).

# Module 3: Generative AI — Hallucination as a Feature
## Lab: In-Silico Binder Design

**Lagos Bio-Design Bootcamp** | [JobAiReady](https://github.com/JobAiReady/lagos-bio-design)

---

### Objective
Design a protein binder that attaches to a specific target surface (e.g., SARS-CoV-2 Spike Protein RBD).

### Prerequisites
- Completed Module 2
- **GPU Runtime enabled** (Runtime → Change runtime type → T4 GPU)
- Target PDB file (we'll download it below)

### Deliverable
A PDB file of the complex showing the binder attached to the target.

> ⚠️ **GPU Required.** Go to Runtime → Change runtime type → T4 GPU.

---
## Background & Key Concepts

**Protein Language Models (PLMs)** are the biological equivalent of GPT — trained on millions of protein sequences to learn the "grammar" of biology. Just as GPT predicts the next word, ESM-2 predicts the next amino acid, building deep representations of structure and function without ever seeing a 3D structure.

**Binder design** is the killer application of generative protein engineering. A binder is a protein designed to physically attach to a specific target — like a custom-made lock for a particular key. In medicine, binders are the basis of antibody drugs, diagnostic reagents, and biosensors. RFDiffusion generates binders by conditioning the diffusion process on a target structure: you specify which surface ("hotspot") to bind, and the model hallucinates a complementary protein.

### "Hallucination as a Feature"

In language models, hallucination = generating false information (a bug). In protein design, hallucination = generating structures that have never existed in nature (a **feature**). The key is *controllable* hallucination: specifying constraints (bind here, fold like this, be this size) while letting the model explore the vast space of possible proteins.

### What You'll Build

```
Target PDB → Select hotspot → RFDiffusion (binder mode) → ProteinMPNN → Binder complex
```

### Key Terms

| Term | Definition |
|------|-----------|
| **Binder** | A protein designed to physically attach to a specific target surface. Basis for antibody drugs and diagnostics |
| **Hotspot Residues** | Amino acid positions on the target surface that contribute most to binding energy — the "anchors" |
| **Interface Energy** | Calculated strength of interaction between binder and target. More negative = stronger binding |
| **pAE (Predicted Aligned Error)** | AlphaFold's confidence about relative positions between chains. Low inter-chain pAE suggests real interaction |
| **PLM (Protein Language Model)** | Neural network trained on millions of sequences to learn the statistical patterns of amino acid usage (e.g., ESM-2) |
| **Diffusion Model** | Generative AI that creates new data by learning to reverse a gradual noising process. Applied to 3D coordinates for proteins |

---
## Setup

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU not found! Enable T4 GPU in Runtime settings.'
print(f'GPU: {torch.cuda.get_device_name(0)}')

!pip install -q biopython matplotlib numpy

In [ ]:
# Install RFDiffusion if not already present
import os, site, torch

if not os.path.exists('/content/RFdiffusion'):
    !git clone https://github.com/RosettaCommons/RFdiffusion.git /content/RFdiffusion
    %cd /content/RFdiffusion

    # Auto-detect CUDA/torch version and install matching DGL
    cuda_ver = torch.version.cuda.replace('.', '')[:3]
    torch_ver = '.'.join(torch.__version__.split('.')[:2])
    wheel_url = f'https://data.dgl.ai/wheels/torch-{torch_ver}/cu{cuda_ver}/repo.html'
    print(f'Installing DGL from {wheel_url}')
    !pip install -q dgl -f {wheel_url}

    # Patch out graphbolt C++ loader (not needed by RFDiffusion, often missing in new wheels)
    for sp in site.getsitepackages():
        gb_init = os.path.join(sp, 'dgl', 'graphbolt', '__init__.py')
        if os.path.exists(gb_init):
            with open(gb_init, 'w') as f:
                f.write('# Patched: graphbolt C++ library not needed for RFDiffusion\n')
            print('Patched DGL graphbolt')
            break

    # Pin torchdata (datapipes removed in newer versions but DGL needs it)
    !pip install -q torchdata==0.7.1

    # Install vendored SE3-Transformer (removed from PyPI)
    !pip install -q -e env/SE3Transformer

    # Install RFDiffusion + missing transitive deps
    !pip install -q -e .
    !pip install -q hydra-core omegaconf pyrsistent

    os.makedirs('models', exist_ok=True)
    !wget -q -P models/ http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt
    !wget -q -P models/ http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt
    print('RFDiffusion installed with binder weights.')
else:
    print('RFDiffusion already installed.')

---
## Step 1: Target Preparation
Download the SARS-CoV-2 Spike RBD structure and isolate the binding site.

**What's happening:** We select a surface region on the target protein where we want our designed binder to attach. These are the 'hotspot' residues.

In [ ]:
# Download Spike RBD structure from PDB
import urllib.request
os.makedirs('/content/targets', exist_ok=True)

# PDB 6M0J: SARS-CoV-2 Spike RBD bound to ACE2
urllib.request.urlretrieve(
    'https://files.rcsb.org/download/6M0J.pdb',
    '/content/targets/6M0J.pdb'
)
print('Downloaded 6M0J.pdb (Spike RBD + ACE2 complex)')

In [ ]:
# Extract just the RBD chain (Chain E in 6M0J)
from Bio.PDB import PDBParser, PDBIO, Select

class ChainSelect(Select):
    def __init__(self, chain_id):
        self.chain_id = chain_id
    def accept_chain(self, chain):
        return chain.get_id() == self.chain_id

parser = PDBParser(QUIET=True)
structure = parser.get_structure('complex', '/content/targets/6M0J.pdb')

io = PDBIO()
io.set_structure(structure)
io.save('/content/targets/spike_rbd.pdb', ChainSelect('E'))

# Count residues
rbd = parser.get_structure('rbd', '/content/targets/spike_rbd.pdb')
n_res = sum(1 for _ in rbd.get_residues())
print(f'Spike RBD extracted: {n_res} residues (Chain E)')

In [ ]:
# Identify hotspot residues (ACE2-binding interface on RBD)
# These are key contact residues from literature
hotspot_residues = [417, 446, 449, 453, 455, 456, 475, 486, 487, 489, 493, 496, 498, 500, 501, 502, 505]
print(f'Hotspot residues ({len(hotspot_residues)}): {hotspot_residues}')
print('These are the ACE2-binding interface residues on the Spike RBD.')

---
## Step 2: Binder Hallucination with RFDiffusion
Run RFDiffusion in 'binder' mode, specifying the target and hotspot residues.

**What's happening:** RFDiffusion generates a completely new protein (50 residues) that is shaped to bind to the Spike RBD surface — it 'hallucinates' a binder from noise.

In [ ]:
os.chdir('/content/RFdiffusion')
os.makedirs('/content/binder_outputs', exist_ok=True)

# Generate 3 binder candidates (50 residues each)
!python scripts/run_inference.py \
    inference.input_pdb=/content/targets/spike_rbd.pdb \
    'contigmap.contigs=[E1-194/0 50-50]' \
    'ppi.hotspot_res=[E417,E486,E498,E501,E505]' \
    inference.output_prefix=/content/binder_outputs/binder \
    inference.model_directory_path=models/ \
    inference.num_designs=3

In [ ]:
# Inspect generated binder complexes
import glob

binder_pdbs = sorted(glob.glob('/content/binder_outputs/binder_*.pdb'))
print(f'Generated {len(binder_pdbs)} binder candidates:\n')

for pdb_path in binder_pdbs:
    structure = parser.get_structure('b', pdb_path)
    chains = list(structure[0].get_chains())
    for chain in chains:
        n_res = sum(1 for _ in chain.get_residues())
        print(f'  {os.path.basename(pdb_path)} — Chain {chain.get_id()}: {n_res} residues')
    print()

---
## Step 3: Interface Analysis & Optimization
Analyze the binding interface to assess how well the binder contacts the target.

In [ ]:
# Analyze interface contacts for each binder
from Bio.PDB import NeighborSearch
import numpy as np

def analyze_interface(pdb_path, target_chain='E', distance_cutoff=8.0):
    structure = parser.get_structure('complex', pdb_path)
    model = structure[0]
    
    target_atoms = [a for a in model[target_chain].get_atoms()]
    binder_chains = [c for c in model.get_chains() if c.get_id() != target_chain]
    
    if not binder_chains:
        return 0, 0
    
    binder_atoms = [a for c in binder_chains for a in c.get_atoms()]
    
    # Find contacts
    ns = NeighborSearch(target_atoms)
    contacts = set()
    for atom in binder_atoms:
        nearby = ns.search(atom.get_vector().get_array(), distance_cutoff)
        for n_atom in nearby:
            contacts.add(n_atom.get_parent().get_id()[1])
    
    return len(contacts), len(binder_atoms)

print('Interface Analysis:')
print(f'{"File":<30} {"Target contacts":>16} {"Binder atoms":>14}')
print('-' * 62)
for pdb_path in binder_pdbs:
    contacts, b_atoms = analyze_interface(pdb_path)
    print(f'{os.path.basename(pdb_path):<30} {contacts:>16} {b_atoms:>14}')

In [ ]:
# Design sequences for the best binder using ProteinMPNN
os.chdir('/content/ProteinMPNN')

best_binder = binder_pdbs[0]  # Use the first candidate
os.makedirs('/content/binder_seqs', exist_ok=True)

!python protein_mpnn_run.py \
    --pdb_path {best_binder} \
    --out_folder /content/binder_seqs \
    --num_seq_per_target 5 \
    --sampling_temp 0.1 \
    --batch_size 1

print('Sequences designed for binder.')

In [ ]:
# Download the best binder complex
from google.colab import files
files.download(best_binder)

---
## Summary

In this lab you:
1. **Prepared a target** — isolated the Spike RBD and identified hotspot residues
2. **Hallucinated binders** — used RFDiffusion to generate de novo proteins shaped to bind the target
3. **Analyzed interfaces** — assessed contact quality at the binding surface
4. **Designed sequences** — used ProteinMPNN to find sequences for the binder backbone

This is the same approach used to design therapeutic antibody alternatives and diagnostic proteins.

**Next:** Module 4 — Applying this pipeline to Lassa Fever (Capstone Project)